# EACF ridge morphology

This tutorial retains the full frequency-lag map, measures whether the candidate resembles a localised oscillation ridge, and calibrates the diagnostics using exact-window signal injections.

In [ ]:
import numpy as np

from urdr import (
    CoherentSignalConfig,
    SegmentSystematicConfig,
    SimulationConfig,
    benchmark_morphology_veto,
    compute_eacf_map,
    eacf_morphology,
    make_observing_window,
    simulate_time_series,
)

In [ ]:
window = make_observing_window(
    duration_days=1.0,
    cadence_seconds=120.0,
    gaps_days=((0.49, 0.51),),
)
simulation = SimulationConfig(
    white_noise_sigma=0.2,
    granulation_amplitude=0.1,
    numax_uhz=1000.0,
    delta_nu_uhz=100.0,
    envelope_width_uhz=400.0,
    oscillation_amplitude=1.5,
)
centres = np.linspace(700.0, 1300.0, 9)
series = simulate_time_series(
    window, simulation, np.random.default_rng(8), include_oscillations=True
)
eacf_map = compute_eacf_map(
    series, centres, 500.0, max_lag_seconds=25_000.0
)
eacf_morphology(
    eacf_map,
    delta_nu_uhz=simulation.delta_nu_uhz,
    expected_numax_uhz=simulation.numax_uhz,
    envelope_width_uhz=simulation.envelope_width_uhz,
)

In [ ]:
benchmark = benchmark_morphology_veto(
    window=window,
    simulation=simulation,
    centre_frequencies_uhz=centres,
    filter_width_uhz=500.0,
    delta_nu_grid_uhz=np.array([90.0, 100.0, 110.0]),
    coherent_contaminants={
        "single_line": CoherentSignalConfig(1000.0, 0.8),
    },
    segment_systematics={
        "variance_jump": [
            SegmentSystematicConfig(0.5, 1.0, amplitude_scale=4.0)
        ],
    },
    realizations=8,
    target_false_positive_rate=0.25,
    target_signal_retention=0.9,
    max_lag_seconds=25_000.0,
    seed=42,
)
{
    "signal_before": benchmark.raw_signal_detection_rate,
    "signal_after": benchmark.accepted_signal_detection_rate,
    "contaminants_before": benchmark.raw_contaminant_detection_rate,
    "contaminants_after": benchmark.accepted_contaminant_detection_rate,
}

The morphology threshold is target- and window-specific. Always report the raw and accepted rates together: rejecting hard negatives is only useful when the loss of genuine oscillator detections remains visible.